# ECG + EHR fusion lab

**CDT in Healthcare Data Science, ML for Time Series.** About 80 minutes.

In the lecture we met two big ideas: aligning modalities in a shared space
(contrastive learning), and combining them for prediction (early, intermediate
and late fusion, with operators such as concatenation, addition, gating and
attention). Today you will try both on ECGs and EHR data from the same
patients in MIMIC-IV.

| Part | Time | What you do |
|---|---|---|
| 1 | 30 min | Train a small CLIP-style ECG and text model from scratch, and compare it with a pretrained ECG encoder |
| 2 | 10 min | Predict 1-year mortality from each modality on its own |
| 3 | 30 min | Fuse the two modalities: where to fuse, and how to combine |
| 4 | 5 min | A quick check on when your features were recorded |

Cells marked **TODO** are for you to complete. Everything else runs as it is.

### Before you start

1. **Data:** download `cdt_fusion_lab_data.zip` from Canvas and put it, still
   zipped, at `MyDrive/cdt_fusion_lab/cdt_fusion_lab_data.zip` in Google Drive
   (a `cdt_fusion_lab` folder at the top level of My Drive).
2. **GPU:** *Runtime > Change runtime type > T4 GPU*. If you can't see the menus,
   use the small arrow next to **Connect** (top right). Without a GPU everything
   still runs, but Part 1 trains for fewer epochs.
3. **Drive access:** the first data cell asks to connect to Google Drive. Sign in
   with the account that holds the zip and allow access. It then unzips the data
   into this session, which takes a few seconds.
4. **Keep your work:** use **Copy to Drive** (or *File > Save a copy in Drive*).
   Changes are not saved back to GitHub.

This data is credentialed: keep it in your own Drive and this session, do not
upload it anywhere else, and do not paste any of it into an AI chatbot.

## Set-up

In [ ]:
import os
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.patches import FancyBboxPatch
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Using device:", device)


def check(description, passed, hint=""):
    """Prints PASS or FAIL for one of the checks after each TODO."""
    print(("PASS  " if passed else "FAIL  ") + description)
    if not passed and hint:
        print("      hint:", hint)

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
    zip_path = Path("/content/drive/MyDrive/cdt_fusion_lab/cdt_fusion_lab_data.zip")
    work_dir = Path("/content")
except ImportError:
    # Running outside Colab: point FUSION_LAB_ZIP at your copy of the zip
    zip_path = Path(os.environ.get("FUSION_LAB_ZIP", "../data/cdt_fusion_lab_data.zip"))
    work_dir = zip_path.parent

data_dir = work_dir / "cdt_fusion_lab_data"
if not data_dir.exists():
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(work_dir)

meta = pd.read_csv(data_dir / "metadata.csv")
ecgs = np.load(data_dir / "ecgs.npy").astype(np.float32)
ecgfounder = np.load(data_dir / "ecgfounder_embeddings.npy")
report_emb = np.load(data_dir / "report_embeddings.npy")
labs = pd.read_csv(data_dir / "labs.csv.gz")
FS = 250

print("ECGs:", ecgs.shape, "| metadata:", meta.shape)
print("ECGFounder embeddings:", ecgfounder.shape, "| report embeddings:", report_emb.shape)
print("lab results:", labs.shape)

Every row is one ECG, and the arrays are in the same order as `metadata.csv`.

- `ecgs`: 12 leads, 10 s at 250 Hz, each recording z-scored.
- `machine_report`: the ECG machine's automatic interpretation (our text modality).
- `age`, `gender`: from the EHR. Every patient was admitted to hospital.
  `ecg_in_ed` is 1 if the ECG was taken in the emergency department (ED),
  before the admission began.
- `labs` (a separate table): every lab result from the patient's hospital
  admission, timed in hours from the ECG. We use it from Part 2.
- `mortality_1_year`: our outcome, death within 365 days of the ECG.
- `ecgfounder_embeddings`: features from a pretrained ECG model (Part 1).
- `report_embeddings`: the machine report encoded by a frozen biomedical language
  model (BioLinkBERT).

Here is what the tabular side looks like: one row per ECG, some EHR fields,
the outcome, and the report text.

In [ ]:
ehr_view = pd.DataFrame({
    "ECG": meta["pair_id"],
    "Patient": meta["subject_id"],
    "Age": meta["age"].round().astype("Int64"),
    "Sex": meta["gender"],
    "ECG taken": meta["ecg_in_ed"].map({1: "In ED, before admission", 0: "During admission"}),
    "Died within 1 year": meta["mortality_1_year"].map({1: "Yes", 0: "No"}),
    "Machine report": meta["machine_report"],
})
ehr_view.head(8).style.hide(axis="index")

In [ ]:
LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]


def plot_ecg(i):
    t = np.arange(ecgs.shape[-1]) / FS
    fig, axes = plt.subplots(6, 2, figsize=(12, 7), sharex=True)
    for lead, ax in enumerate(axes.T.flat):
        ax.plot(t, ecgs[i, lead], lw=0.7)
        ax.set_ylabel(LEADS[lead], rotation=0, labelpad=15)
        ax.set_yticks([])
    row = meta.iloc[i]
    fig.suptitle(f"{row.age:.0f}-year-old {row.gender}: {row.machine_report}", fontsize=10)
    axes[-1, 0].set_xlabel("Time (s)")
    axes[-1, 1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()


plot_ecg(0)
print(meta[["age", "ecg_in_ed", "mortality_1_year"]].describe().round(2))

The `labs` table has one row per result, from the whole hospital admission.
`hours_from_ecg` is negative before the ECG and positive after it. Below are
the most common tests, then one patient's results around their ECG. The shaded
side is after the ECG: we will not know it when we make a prediction. Try
`plot_labs` with other ECG numbers.

In [ ]:
PLOT_ITEMS = {50912: "Creatinine (mg/dL)", 50971: "Potassium (mmol/L)",
              51222: "Haemoglobin (g/dL)", 51301: "White cells (K/uL)"}


def plot_labs(pair_id, items=PLOT_ITEMS, hours=72):
    rows = labs[(labs["pair_id"] == pair_id) & labs["hours_from_ecg"].between(-hours, hours)]
    fig, axes = plt.subplots(1, len(items), figsize=(13, 2.6), sharex=True)
    for ax, (itemid, name) in zip(axes, items.items()):
        r = rows[rows["itemid"] == itemid]
        ax.plot(r["hours_from_ecg"], r["value"], marker="o", ms=3)
        ax.axvline(0, color="#d63a3a", lw=1)
        ax.axvspan(0, hours, color="grey", alpha=0.15)
        ax.set_xlim(-hours, hours)
        ax.set_title(name, fontsize=9)
        ax.set_xlabel("Hours from ECG")
    fig.suptitle(f"ECG {pair_id}: lab results (red line: the ECG)", fontsize=10)
    plt.tight_layout()
    plt.show()


print("Most common tests:")
print(labs["test"].value_counts().head(8).to_string())
around = labs[labs["itemid"].isin(PLOT_ITEMS) & labs["hours_from_ecg"].between(-72, 72)]
example = around["pair_id"].value_counts().index[0]
plot_labs(example)
labs[(labs["pair_id"] == example) & labs["hours_from_ecg"].between(-24, 0)].head(10).style.hide(axis="index")

## Part 1: aligning ECGs with text (30 min)

CLIP ([Radford et al., ICML 2021](https://arxiv.org/abs/2103.00020)) learns a
shared space for two modalities by pulling each example towards its own partner
and away from everyone else's in the batch. It was designed for images and
captions; [MERL (Liu et al., ICML 2024)](https://arxiv.org/abs/2403.06659) is
a recent example of the same idea for ECGs and their reports.

We use the same set-up as a recent ECG and text pipeline (Jinpei Han's, MIT
licence): a 1D ConvNeXtV2 ECG encoder, a **frozen** text encoder, a linear
projection for each, and the symmetric InfoNCE loss.

The text encoder is frozen, so its embeddings never change. We computed them
once in advance (`report_embeddings`), which saves loading a 400 MB language
model today.

We hold out 20% of **patients** to test on.

In [ ]:
split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(split.split(ecgs, groups=meta["subject_id"]))
print(f"train: {len(train_idx)} ECGs, test: {len(test_idx)} ECGs")

The ECG encoder is **ConvNeXt V2**
([Woo et al., CVPR 2023](https://arxiv.org/abs/2301.00808)), a modern
convolutional network for images. It builds on ConvNeXt
([Liu et al., CVPR 2022](https://arxiv.org/abs/2201.03545)) and adds a "global
response normalisation" (GRN) layer. We use its smallest size ("atto", about
3 M parameters) with every 2D convolution swapped for a 1D one, so it slides
along time across the 12 leads. You do not need to change this cell.

In [ ]:
class LayerNorm1d(nn.Module):
    """LayerNorm over channels for (batch, channels, length) tensors."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(1, dim, 1))
        self.bias = nn.Parameter(torch.zeros(1, dim, 1))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(1, keepdim=True)
        var = x.var(1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(var + self.eps) * self.weight + self.bias


class GRN(nn.Module):
    """Global response normalisation from ConvNeXtV2."""

    def __init__(self, dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1, 1, dim))
        self.beta = nn.Parameter(torch.zeros(1, 1, dim))

    def forward(self, x):
        gx = torch.norm(x, p=2, dim=1, keepdim=True)
        nx = gx / (gx.mean(dim=-1, keepdim=True) + 1e-6)
        return self.gamma * (x * nx) + self.beta + x


class Block(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.grn = GRN(4 * dim)
        self.pwconv2 = nn.Linear(4 * dim, dim)

    def forward(self, x):
        y = self.dwconv(x).permute(0, 2, 1)
        y = self.pwconv2(self.grn(self.act(self.pwconv1(self.norm(y)))))
        return x + y.permute(0, 2, 1)


class ConvNeXtV2(nn.Module):
    def __init__(self, in_chans=12, depths=(2, 2, 6, 2), dims=(40, 80, 160, 320)):
        super().__init__()
        self.downsample = nn.ModuleList([nn.Sequential(nn.Conv1d(in_chans, dims[0], 4, stride=4), LayerNorm1d(dims[0]))])
        for i in range(3):
            self.downsample.append(nn.Sequential(LayerNorm1d(dims[i]), nn.Conv1d(dims[i], dims[i + 1], 2, stride=2)))
        self.stages = nn.ModuleList([nn.Sequential(*[Block(dims[i]) for _ in range(depths[i])]) for i in range(4)])
        self.norm = nn.LayerNorm(dims[-1], eps=1e-6)
        self.out_dim = dims[-1]

    def forward(self, x):
        for down, stage in zip(self.downsample, self.stages):
            x = stage(down(x))
        return self.norm(x.mean(-1))


class ECGTextCLIP(nn.Module):
    """ECG encoder plus a linear projection for each modality into a shared space."""

    def __init__(self, ecg_encoder, ecg_dim, text_dim=768, shared_dim=128):
        super().__init__()
        self.ecg_encoder = ecg_encoder
        self.ecg_proj = nn.Linear(ecg_dim, shared_dim, bias=False)
        self.text_proj = nn.Linear(text_dim, shared_dim, bias=False)

    def forward(self, ecg, text_emb):
        z_ecg = F.normalize(self.ecg_proj(self.ecg_encoder(ecg)), dim=-1)
        z_text = F.normalize(self.text_proj(text_emb), dim=-1)
        return z_ecg, z_text


print(f"ECG encoder parameters: {sum(p.numel() for p in ConvNeXtV2().parameters()) / 1e6:.1f} M")

### How the CLIP loss works

Take a batch of $N$ ECG and report pairs. After projection and L2
normalisation, ECG $i$ has embedding $\mathbf{e}_i$ and report $j$ has
$\mathbf{t}_j$. Their dot product is the cosine similarity, and we scale it by
a temperature $\tau$ (here 0.07):

$$s_{ij} = \frac{\mathbf{e}_i \cdot \mathbf{t}_j}{\tau}$$

This gives an $N \times N$ matrix $S$. The diagonal holds the true pairs and
everything else is a mismatched pair (a "negative").

Now read **row $i$** as a classification problem: "which of the $N$ reports
belongs to ECG $i$?" The scores $s_{i1}, \dots, s_{iN}$ are the logits, and the
correct class is $i$. Its loss is the usual softmax cross-entropy:

$$L_{\text{ECG}\to\text{text}} = -\frac{1}{N}\sum_{i=1}^{N}
  \log \frac{\exp(s_{ii})}{\sum_{j=1}^{N} \exp(s_{ij})}$$

Reading the **columns** the same way ("which ECG belongs to report $j$?") gives
$L_{\text{text}\to\text{ECG}}$, which is the same formula applied
to $S^\top$. The CLIP loss is the average of the two:

$$L = \tfrac{1}{2}\left(L_{\text{ECG}\to\text{text}} +
  L_{\text{text}\to\text{ECG}}\right)$$

This is the **InfoNCE** loss
([van den Oord et al., 2018](https://arxiv.org/abs/1807.03748)), applied in
both directions. A small $\tau$ sharpens the softmax, so the model is pushed
harder to separate the true pair from close negatives.

### TODO 1: the CLIP loss

Write `clip_loss`. `z_ecg` and `z_text` are `(N, dim)` tensors, already
normalised, and row `i` of each comes from the same ECG. Four steps:

1. **The similarity matrix $S$**, shape `(N, N)`, with $s_{ij} = \mathbf{e}_i \cdot
   \mathbf{t}_j / \tau$. In PyTorch, `a @ b.T` gives the dot product of every
   row of `a` with every row of `b`.
2. **The targets:** for row $i$, which column is the right answer? Make a tensor
   holding that answer for every row. `torch.arange(N, device=z_ecg.device)`
   gives `[0, 1, ..., N-1]`.
3. **ECG to text:** `F.cross_entropy(logits, targets)` takes a matrix of raw
   scores (no softmax needed) and one correct column index per row. For each row
   it applies the softmax, takes minus the log of the probability in the correct
   column, and averages over rows. That is exactly the formula for
   $L_{\text{ECG}\to\text{text}}$ above.
4. **Text to ECG:** the same on the columns of $S$ (transpose it, `.T`). Return
   the average of the two.

In [ ]:
def clip_loss(z_ecg, z_text, temperature=0.07):
    # TODO: your code here
    raise NotImplementedError("Complete the TODO above")


N = 8
one_point = F.normalize(torch.ones(N, 16), dim=-1)
check("everything at one point gives the chance-level loss",
      abs(clip_loss(one_point, one_point).item() - np.log(N)) < 1e-4,
      "if all N scores in a row are equal, the softmax gives each column probability 1/N")
perfect = torch.eye(N)
expected = np.log(1 + (N - 1) * np.exp(-1 / 0.07))
check("perfectly matched pairs give exactly the right loss",
      abs(clip_loss(perfect, perfect).item() - expected) < 1e-4,
      "divide the dot products by the temperature, and the right column for row i is i")
a, b = F.normalize(torch.randn(N, 16), dim=-1), F.normalize(torch.randn(N, 16), dim=-1)
check("ECG and text are treated the same way",
      abs(clip_loss(a, b).item() - clip_loss(b, a).item()) < 1e-5,
      "average the loss over the rows (ECG to text) and over the columns (text to ECG)")

### Measuring the shared space: retrieval

For each of the $n$ test ECGs (about 200), rank all $n$ test reports by
similarity to it. **Recall at $k$ (R@$k$)** is the fraction of test ECGs whose own report is
in the top $k$. R@1 asks for the right report first; R@10 asks for it anywhere
in the top 10.

One complication: many reports are word-for-word identical ("Sinus rhythm
Normal ECG" appears dozens of times). If ECG $i$'s report is shared by $m_i$
test ECGs (counting itself), retrieving *any* of those $m_i$ reports is
equally correct, so we count it as a hit.

**Where the chance level comes from.** Suppose the model ranked reports at
random. The top $k$ is then a random draw of $k$ reports without replacement,
and the chance of missing all $m_i$ matching reports is

$$P(\text{miss}_i) = \prod_{j=0}^{k-1} \frac{n - m_i - j}{n - j}$$

So random ranking gives R@$k$ $= \frac{1}{n}\sum_i \left(1 - P(\text{miss}_i)\right)$.

With $n = 200$:

- **A unique report** ($m_i = 1$) gets a chance R@1 of $1/200 = 0.005$.
- **A report shared by 20 test ECGs** gets $20/200 = 0.1$.

Averaging over the test set gives chance R@1 of about 0.02. A model is only
learning something if it beats these numbers.

In [ ]:
test_texts = meta["machine_report"].to_numpy()[test_idx]


def retrieval(z_ecg, z_text, texts, ks=(1, 5, 10)):
    texts = np.asarray(texts)
    order = (z_ecg @ z_text.T).argsort(dim=1, descending=True).cpu().numpy()
    n = len(texts)
    same = np.array([(texts == t).sum() for t in texts])
    result = {}
    for k in ks:
        hits = np.mean([texts[i] in set(texts[order[i, :k]]) for i in range(n)])
        chance = np.mean([1 - np.prod([(n - s - j) / (n - j) for j in range(k)]) for s in same])
        result[f"R@{k}"] = (round(float(hits), 3), round(float(chance), 3))
    return result


@torch.no_grad()
def evaluate_clip(model, ecg_inputs, ks=(1, 5, 10)):
    model.eval()
    z_ecg, z_text = model(torch.tensor(ecg_inputs[test_idx]).to(device), torch.tensor(report_emb[test_idx]).to(device))
    return retrieval(z_ecg, z_text, test_texts, ks)


def train_clip(model, ecg_inputs, epochs, lr, batch_size=32, eval_every=None, temperature=0.07):
    """Train on the training split. ecg_inputs is either raw ECGs or fixed embeddings.

    With eval_every, also records test R@5 every few epochs.
    """
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    X = torch.tensor(ecg_inputs[train_idx])
    T = torch.tensor(report_emb[train_idx])
    losses, curve = [], []
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for batch in torch.randperm(len(X)).split(batch_size):
            z_ecg, z_text = model(X[batch].to(device), T[batch].to(device))
            loss = clip_loss(z_ecg, z_text, temperature)
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * len(batch)
        losses.append(epoch_loss / len(X))
        if eval_every and epoch % eval_every == 0:
            curve.append((epoch, evaluate_clip(model, ecg_inputs, ks=(5,))["R@5"][0]))
    return losses, curve

### Training from scratch

Now train the ConvNeXtV2 from scratch on the 800 training ECGs, checking test
retrieval as it goes. Real ECG foundation models train on millions of ECGs.

**Experiment.** The settings are in the cell below. Change them and re-run this
cell, the ECGFounder cell and the plot: each run starts again from random
weights. Things to try:

- **More epochs.** Is R@5 still rising at the end, or has it levelled off?
  Does training loss keep falling while test retrieval does not (overfitting)?
- **Batch size** (64 or 128). Each ECG then competes against more negatives,
  but there are fewer update steps per epoch.
- **Learning rate**, and the **temperature** $\tau$ in the loss.

The linear probes further down use whichever model you trained last.

In [ ]:
# About 1 minute on a T4 GPU for 40 epochs. Without a GPU each epoch takes about
# 30 s, so we default to fewer.
EPOCHS = 40 if device != "cpu" else 8
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
TEMPERATURE = 0.07
EVAL_EVERY = 5 if device != "cpu" else 2  # how often to check test R@5, in epochs

torch.manual_seed(SEED)
scratch = ECGTextCLIP(ConvNeXtV2(), ecg_dim=320)
start = time.time()
scratch_losses, scratch_curve = train_clip(scratch, ecgs, epochs=EPOCHS, lr=LEARNING_RATE,
                                           batch_size=BATCH_SIZE, eval_every=EVAL_EVERY,
                                           temperature=TEMPERATURE)
print(f"trained in {time.time() - start:.0f} s")
print("From scratch, test retrieval (score, chance):", evaluate_clip(scratch, ecgs))

### Here's one I made earlier

`ecgfounder_embeddings` come from **ECGFounder** (NEJM AI 2025), a model trained
on more than 10 million ECGs. We keep it frozen and train only the two linear
projections, with exactly the same loss. That takes seconds.

In [ ]:
class Frozen(nn.Module):
    """Passes pre-computed embeddings straight through."""

    def forward(self, x):
        return x


torch.manual_seed(SEED)
founder_clip = ECGTextCLIP(Frozen(), ecg_dim=ecgfounder.shape[1])
founder_losses, _ = train_clip(founder_clip, ecgfounder, epochs=100, lr=1e-3, batch_size=64)
founder_retrieval = evaluate_clip(founder_clip, ecgfounder)
print("ECGFounder + linear heads, test retrieval (score, chance):", founder_retrieval)

In [ ]:
epochs_seen, r5 = zip(*scratch_curve) if scratch_curve else ([], [])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(scratch_losses, label="from scratch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training loss")
axes[0].legend()
axes[1].plot(epochs_seen, r5, marker="o", label="from scratch")
axes[1].axhline(founder_retrieval["R@5"][0], color="#1baf7a", label="ECGFounder + linear heads")
axes[1].axhline(founder_retrieval["R@5"][1], color="grey", ls="--", label="chance")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Test R@5")
axes[1].legend()
plt.tight_layout()
plt.show()

### What did each encoder learn about the heart?

Retrieval only tells us about matching reports. A **linear probe** asks a more
direct question: if we freeze an encoder and fit a logistic regression on its
embeddings, how well can it detect a condition? We compare three encoders:

- an **untrained** ConvNeXtV2 (random weights: the floor);
- the ConvNeXtV2 you just **trained from scratch**;
- **ECGFounder**.

The labels come from the machine report: atrial fibrillation, and "abnormal ECG".

In [ ]:
@torch.no_grad()
def encode(encoder, X, batch_size=100):
    encoder.to(device).eval()
    out = [encoder(torch.tensor(X[i:i + batch_size]).to(device)).cpu() for i in range(0, len(X), batch_size)]
    return torch.cat(out).numpy()


def linear_probe(embeddings, labels, C=0.01):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    model.fit(embeddings[train_idx], labels[train_idx])
    return roc_auc_score(labels[test_idx], model.predict_proba(embeddings[test_idx])[:, 1])


torch.manual_seed(SEED)
encoders = {
    "untrained": encode(ConvNeXtV2(), ecgs),
    "from scratch": encode(scratch.ecg_encoder, ecgs),
    "ECGFounder": ecgfounder,
}
probe_labels = {
    "atrial fibrillation": meta["machine_report"].str.contains("atrial fibrillation", case=False).to_numpy().astype(int),
    "abnormal ECG": meta["machine_report"].str.contains("abnormal ecg", case=False).to_numpy().astype(int),
}
for task, labels in probe_labels.items():
    print(f"{task} ({labels[test_idx].sum()} positives in the test set)")
    for name, emb in encoders.items():
        print(f"   {name:13s} AUROC {linear_probe(emb, labels):.3f}")

**Questions for Part 1**

1. How does your from-scratch model compare with the untrained encoder and with
   ECGFounder, on retrieval and on the probes? Is it still improving at 40 epochs?
2. Give two reasons why 800 pairs is not enough to train an ECG encoder this way.
   Think about the data and about how many negatives each example sees.
3. Even the untrained encoder does better than chance on the probes. Why might
   random convolutional features carry any information at all?

*Your answers:*

## Part 2: one modality at a time (10 min)

Task: **does the patient die within a year of the ECG?** We predict at the
moment the ECG is taken, so every feature must be known by then. From here on
we use ECGFounder for the ECG. We compare:

- **EHR only:** age, sex and nine routine blood tests (kidney function,
  electrolytes, full blood count and glucose). For each test we take the most
  recent result in the 24 hours **before** the ECG.
- **ECG only:** the ECGFounder embedding.

`labs` has one row per result, with `hours_from_ecg` (negative means before the
ECG). It covers the hospital admission only: tests done in the ED are not in
this extract. The function below picks one value per test from a time window.

In [ ]:
LAB_PANEL = {
    50912: "creatinine", 51006: "urea", 50971: "potassium", 50983: "sodium", 50882: "bicarbonate",
    51222: "haemoglobin", 51301: "white_cells", 51265: "platelets", 50931: "glucose",
}


def last_lab_values(start_hours, end_hours):
    """Most recent result of each panel test from start_hours to end_hours around the ECG (NaN if none)."""
    window = labs[labs["itemid"].isin(LAB_PANEL) & labs["hours_from_ecg"].between(start_hours, end_hours)]
    last = window.sort_values("hours_from_ecg").groupby(["pair_id", "itemid"])["value"].last().unstack()
    return last.reindex(index=meta["pair_id"], columns=list(LAB_PANEL)).rename(columns=LAB_PANEL).reset_index(drop=True)


lab_values = last_lab_values(-24, 0)
no_recent_labs = lab_values.isna().all(axis=1)
print(f"{(~no_recent_labs).sum()} of {len(meta)} ECGs have at least one panel test in the 24 h before")
lab_values.describe().T[["count", "min", "50%", "max"]].round(1)

Each model fills a missing result with the median of its training fold, and
`no_recent_labs` marks the patients with no recent tests at all.

Each model is a logistic regression. We use 5-fold cross-validation and report
the pooled out-of-fold AUROC with a bootstrap 95% confidence interval.

In [ ]:
y = meta["mortality_1_year"].to_numpy()
groups = meta["subject_id"].to_numpy()
X_ehr = np.column_stack([
    meta["age"],
    (meta["gender"] == "Male").astype(float),
    lab_values,
    no_recent_labs.astype(float),
])
N_EHR = X_ehr.shape[1]
X_ecg = ecgfounder
print(f"{len(y)} ECGs, {y.sum()} deaths within a year ({y.mean():.0%})")
print(f"{meta['subject_id'].nunique()} patients; some have more than one ECG")
print(f"{N_EHR} EHR features, {X_ecg.shape[1]} ECG features")

Some patients have more than one ECG. If one patient's ECGs ended up on both
sides of a split, the model could recognise the patient rather than learn the
task, so the folds keep **all of a patient's ECGs together** (and the same
proportion of deaths in each fold).

In [ ]:
FOLDS = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_ehr, y, groups))


def cv_predict(X, C, preprocess=None):
    """Out-of-fold predicted probabilities from a logistic regression."""
    oof = np.zeros(len(y))
    for train, test in FOLDS:
        X_train, X_test = X[train], X[test]
        if preprocess is not None:
            X_train, X_test = preprocess(X_train, X_test, y[train])
        model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                              LogisticRegression(C=C, max_iter=5000))
        model.fit(X_train, y[train])
        oof[test] = model.predict_proba(X_test)[:, 1]
    return oof

### TODO 2: how sure are we? A bootstrap confidence interval

An AUROC from 1,000 ECGs is an estimate: another 1,000 patients would give a
different number. Before comparing models we need to know how big that
difference could be. The **bootstrap** estimates it from the data we have:

1. Draw a resample: $n$ rows chosen **with replacement** from the $n$ ECGs.
2. Compute the AUROC on the resample.
3. Repeat many times. The middle 95% of those AUROCs is the interval.

Write `auroc_ci(y_true, p)`, returning `(auroc, lower, upper)`. Useful tools:

- `rng.integers(0, n, n)` draws `n` row indices from `0` to `n - 1`, with
  replacement.
- `roc_auc_score(y, p)` computes the AUROC. A resample that happens to contain
  only one class has no AUROC, so skip it.
- `np.percentile(values, [2.5, 97.5])` gives the two ends.

In [ ]:
def auroc_ci(y_true, p, n_boot=1000):
    # TODO: your code here
    raise NotImplementedError("Complete the TODO above")
    return roc_auc_score(y_true, p), lower, upper


def expected_width(y_true, p):
    """Width of a 95% interval from the standard-error formula of Hanley and McNeil (1982)."""
    a, n1, n0 = roc_auc_score(y_true, p), y_true.sum(), len(y_true) - y_true.sum()
    q1, q2 = a / (2 - a), 2 * a ** 2 / (1 + a)
    return 2 * 1.96 * np.sqrt((a * (1 - a) + (n1 - 1) * (q1 - a ** 2) + (n0 - 1) * (q2 - a ** 2)) / (n1 * n0))


toy_y = rng.integers(0, 2, 2000)
toy_p = toy_y + rng.normal(0, 1.5, 2000)
for n in (2000, 200):
    auc, lo, hi = auroc_ci(toy_y[:n], toy_p[:n])
    ratio = (hi - lo) / expected_width(toy_y[:n], toy_p[:n])
    check(f"{n} ECGs: the interval contains the AUROC", lo <= auc <= hi)
    check(f"{n} ECGs: the interval is about the width theory predicts (yours: {ratio:.2f} times)",
          0.75 < ratio < 1.3,
          "each resample has as many rows as the data, drawn with replacement")

In [ ]:
results = {}


def record(name, p):
    results[name] = auroc_ci(y, p)
    auc, lo, hi = results[name]
    print(f"{name:40s} AUROC {auc:.3f}  [{lo:.3f}, {hi:.3f}]")

### Regularisation strength `C`

scikit-learn's `LogisticRegression` finds the weights $\mathbf{w}$ and bias $b$
by minimising, with labels $y_i \in \{-1, +1\}$:

$$\underbrace{\tfrac{1}{2}\lVert \mathbf{w} \rVert^2}_{\text{penalty on large weights}}
  \; + \; C \sum_{i} \underbrace{\log\left(1 + e^{-y_i (\mathbf{w}^\top \mathbf{x}_i + b)}\right)}_{\text{fit to the data (log-loss)}}$$

$C$ sets how much the data-fit term counts relative to the penalty. Dividing by
$C$ shows it is the same as a penalty of strength $\lambda = 1/C$. So a
**smaller $C$** means the penalty dominates and the weights are pulled towards
zero: **stronger regularisation**. Large $C$ lets the model fit the training
data closely, which risks overfitting.

This matters here: the ECG model has 1,024 features but only about 800
training ECGs per fold, so it can fit noise easily. The EHR model has 12
features and barely needs regularising.

Run the sweep below to see the effect for the ECG model.

In [ ]:
C_VALUES = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
sweep = [roc_auc_score(y, cv_predict(X_ecg, C=C)) for C in C_VALUES]
plt.semilogx(C_VALUES, sweep, marker="o")
plt.xlabel("C (smaller = stronger regularisation)")
plt.ylabel("ECG only AUROC")
plt.show()

We use the values below. Change them (and the ones in later cells) and see
what happens.

Be aware that picking `C` by looking at these same cross-validated scores is
slightly optimistic, because the test folds are helping to choose `C`. The
careful way is to choose `C` inside each training fold (nested
cross-validation), which is one of the extensions at the end.

In [ ]:
C_EHR = 1.0
C_ECG = 0.001
p_ehr = cv_predict(X_ehr, C=C_EHR)
record("EHR only", p_ehr)
p_ecg = cv_predict(X_ecg, C=C_ECG)
record("ECG only (ECGFounder)", p_ecg)

**Questions for Part 2**

1. Which modality is stronger? Are the confidence intervals separated?
2. Compare the death rate of patients with and without recent labs
   (`meta.groupby(no_recent_labs)["mortality_1_year"].mean()`). Who are the
   patients without labs? (Try `pd.crosstab(no_recent_labs, meta["ecg_in_ed"])`.)
   What could the model learn from `no_recent_labs`?
3. Your bootstrap resamples the predictions while keeping the models fixed.
   Which sources of uncertainty does it leave out?

*Your answers:*

## Part 3: fusion (30 min)

Two decisions from the lecture: **where** in the pipeline to fuse, and **how**
to combine the modalities once you do.

In [ ]:
# @title Diagram: where to fuse { display-mode: "form" }
EHR_C, ECG_C, FUSED_C, MODEL_C = "#eb6834", "#2a78d6", "#1baf7a", "#d9d8d3"


def _box(ax, x, y, w, h, text, color, text_color="white"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08", fc=color, ec="none"))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=9, color=text_color)


def _arrow(ax, x0, y0, x1, y1):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", color="#52514e", lw=1))


def _vector(ax, x, y, values, color, cell=0.18, h=0.3):
    for i, v in enumerate(values):
        ax.add_patch(plt.Rectangle((x + i * cell, y), cell * 0.92, h, fc=color, alpha=0.25 + 0.75 * v, ec="none"))
    return x + len(values) * cell


def _inputs(ax, y):
    _box(ax, 2.0, y + 0.1, 1.1, 0.4, "EHR", EHR_C)
    _box(ax, 2.0, y - 0.5, 1.1, 0.4, "ECG", ECG_C)


def _output(ax, x, y):
    _arrow(ax, x, y, 8.5, y)
    ax.text(8.6, y, "prediction", va="center", fontsize=9)


fig, ax = plt.subplots(figsize=(11, 4.2))
ax.set_xlim(0, 11)
ax.set_ylim(0, 4.2)
ax.axis("off")
for name, yc in {"Early fusion": 3.3, "Intermediate fusion": 1.95, "Late fusion": 0.6}.items():
    ax.text(0.0, yc, name, ha="left", va="center", fontsize=10, weight="bold")

y0 = 3.3
_inputs(ax, y0)
_box(ax, 4.0, y0 - 0.25, 1.4, 0.5, "join features", FUSED_C)
_box(ax, 6.3, y0 - 0.25, 1.4, 0.5, "one model", MODEL_C, "black")
_arrow(ax, 3.1, y0 + 0.3, 4.0, y0)
_arrow(ax, 3.1, y0 - 0.3, 4.0, y0)
_arrow(ax, 5.4, y0, 6.3, y0)
_output(ax, 7.7, y0)

y0 = 1.95
_inputs(ax, y0)
_box(ax, 3.5, y0 + 0.1, 1.2, 0.4, "encoder", MODEL_C, "black")
_box(ax, 3.5, y0 - 0.5, 1.2, 0.4, "encoder", MODEL_C, "black")
_box(ax, 5.2, y0 - 0.25, 1.4, 0.5, "combine", FUSED_C)
_box(ax, 7.0, y0 - 0.25, 0.9, 0.5, "head", MODEL_C, "black")
for dy in (0.3, -0.3):
    _arrow(ax, 3.1, y0 + dy, 3.5, y0 + dy)
    _arrow(ax, 4.7, y0 + dy, 5.2, y0)
_arrow(ax, 6.6, y0, 7.0, y0)
_output(ax, 7.9, y0)
ax.text(5.9, y0 - 0.55, "concatenate, add,\ngate or attend", ha="center", va="top", fontsize=8, color="#52514e")

y0 = 0.6
_inputs(ax, y0)
_box(ax, 3.5, y0 + 0.1, 1.6, 0.4, "EHR model", MODEL_C, "black")
_box(ax, 3.5, y0 - 0.5, 1.6, 0.4, "ECG model", MODEL_C, "black")
_box(ax, 6.0, y0 - 0.25, 1.9, 0.5, "combine predictions", FUSED_C)
for dy in (0.3, -0.3):
    _arrow(ax, 3.1, y0 + dy, 3.5, y0 + dy)
    _arrow(ax, 5.1, y0 + dy, 6.0, y0)
_output(ax, 7.9, y0)
plt.tight_layout()
plt.show()

### 3a. Where to fuse: early or late

**Early fusion:** concatenate the features and fit one model.
**Late fusion:** fit one model per modality and combine their predictions.

We start with the simplest version of each. They are not the only ones, so
once the TODOs below work, try your own (and `record` them so they appear in
the summary):

- **Late fusion:**
  - a weighted average $w \, p_{\text{EHR}} + (1 - w) \, p_{\text{ECG}}$;
  - averaging the log-odds instead of the probabilities;
  - the maximum of the two.
- **Early fusion:**
  - a different way, or size, of shrinking the ECG block (TODO 4);
  - a different `C`;
  - scaling the EHR block up so it is not drowned out.

In [ ]:
X_both = np.column_stack([X_ehr, X_ecg])
p_early_naive = cv_predict(X_both, C=C_ECG)
record("Early fusion (concatenate)", p_early_naive)

### TODO 3: late fusion

Combine `p_ehr` and `p_ecg` into one prediction. Start with the simplest late
fusion, the average of the two probabilities, then try the variants above.

In [ ]:
# TODO: your code here
raise NotImplementedError("Complete the TODO above")
check("one probability per ECG, between 0 and 1",
      np.shape(p_late) == y.shape and 0 <= np.min(p_late) and np.max(p_late) <= 1)
record("Late fusion (average)", p_late)

Late fusion beats both single models. To see why, split the ECGs by whether
the patient had recent labs:

In [ ]:
for name, rows in [("recent labs", ~no_recent_labs), ("no recent labs", no_recent_labs)]:
    r = rows.to_numpy()
    aucs = [roc_auc_score(y[r], p[r]) for p in (p_ehr, p_ecg, p_late)]
    print(f"{name:15s} ({r.sum()} ECGs)  EHR %.3f  ECG %.3f  late %.3f" % tuple(aucs))

**Discuss:** which modality matters more for whom? Why can an average of two
models beat both of them? A fixed average gives every patient the same mix.
What would a better fusion do instead?

*Your answers:*

Early fusion sees strictly more information than either model alone, yet it
barely beats the EHR model, while late fusion gains about 0.05. Before you read
on, discuss with your neighbour why that might be.

Hint: there are 12 EHR columns and 1,024 ECG columns, and a single `C` for all
of them. The strong regularisation the ECG block needs also shrinks the EHR
coefficients.

### TODO 4: make early fusion work

Write `shrink_ecg_block` so that early fusion sees the 12 EHR columns plus a
much smaller ECG block (**at most 64 columns**), so that we can use a weaker
`C`. How you shrink it is up to you. Some ideas:

- **PCA:** keep the top principal components (`sklearn.decomposition.PCA`).
- **Random projection:** multiply by a fixed random matrix (set a seed).
- **Supervised selection:** keep the ECG columns most associated with the
  outcome, using `y_train`.
- **A single risk score:** the log-odds from an ECG-only model. Careful: the
  training rows then need out-of-fold scores. Why?

Standardise the ECG columns first if your method cares about scale. The one
rule: **fit everything on the training fold only**. The function is called once
per fold, and the check below changes the test rows to make sure your training
output does not move.

In [ ]:
def shrink_ecg_block(X_train, X_test, y_train):
    ehr_train, ehr_test = X_train[:, :N_EHR], X_test[:, :N_EHR]
    ecg_train, ecg_test = X_train[:, N_EHR:], X_test[:, N_EHR:]
    # TODO: your code here
    raise NotImplementedError("Complete the TODO above")
    return np.column_stack([ehr_train, ecg_train]), np.column_stack([ehr_test, ecg_test])


train, test = FOLDS[0]
out_train, out_test = shrink_ecg_block(X_both[train], X_both[test], y[train])
moved_test = X_both[test] + rng.normal(0, 1, X_both[test].shape)
again_train, _ = shrink_ecg_block(X_both[train], moved_test, y[train])
check("one row out for every row in", len(out_train) == len(train) and len(out_test) == len(test))
check("EHR columns passed through unchanged",
      np.array_equal(out_train[:, :N_EHR], X_both[train][:, :N_EHR], equal_nan=True))
check(f"ECG block has at most 64 columns (yours has {out_train.shape[1] - N_EHR})",
      out_train.shape[1] - N_EHR <= 64)
check("nothing was fitted on the test rows", np.allclose(out_train, again_train, equal_nan=True),
      "fit scalers, projections or models on X_train only, and fix any random seed")

p_early_shrunk = cv_predict(X_both, C=1.0, preprocess=shrink_ecg_block)
record("Early fusion (EHR + shrunk ECG)", p_early_shrunk)

### 3b. How to combine: intermediate fusion

Intermediate fusion encodes each modality into an embedding of the same size,
`h_EHR` and `h_ECG`, then combines them before a shared prediction head. The
lecture covered four ways to combine them:

In [ ]:
# @title Diagram: how to combine two embeddings { display-mode: "form" }
demo = np.random.default_rng(1)
h_e, h_c = demo.uniform(0, 1, 5), demo.uniform(0, 1, 5)
fig, axes = plt.subplots(1, 5, figsize=(13, 3.2))
for ax, title in zip(axes, ["Concatenation", "Addition", "Gating", "Attention", "Shared space (learnt)"]):
    ax.set_xlim(0, 2.2)
    ax.set_ylim(0, 3)
    ax.axis("off")
    ax.set_title(title, fontsize=10)
for ax in axes[:3]:
    _vector(ax, 0.2, 2.3, h_e, EHR_C)
    _vector(ax, 0.2, 1.7, h_c, ECG_C)

ax = axes[0]
end = _vector(ax, 0.2, 0.6, h_e, EHR_C)
_vector(ax, end, 0.6, h_c, ECG_C)
ax.text(1.1, 1.2, "[ h_EHR , h_ECG ]", ha="center", fontsize=8)
ax.text(1.1, 0.25, "keeps everything;\ntwice as long", ha="center", va="top", fontsize=8, color="#52514e")

ax = axes[1]
_vector(ax, 0.2, 0.6, (h_e + h_c) / 2, FUSED_C)
ax.text(1.1, 1.2, "h_EHR + h_ECG", ha="center", fontsize=8)
ax.text(1.1, 0.25, "same size;\nmodalities must align", ha="center", va="top", fontsize=8, color="#52514e")

ax = axes[2]
gate_demo = demo.uniform(0, 1, 5)
_vector(ax, 0.2, 1.15, gate_demo, "#52514e", h=0.2)
_vector(ax, 0.2, 0.6, gate_demo * h_e + (1 - gate_demo) * h_c, FUSED_C)
ax.text(1.1, 1.45, "gate g in [0, 1]", ha="center", fontsize=8)
ax.text(1.1, 0.25, "g * h_EHR + (1 - g) * h_ECG\nlearnt, per patient", ha="center", va="top", fontsize=8, color="#52514e")

ax = axes[3]
_box(ax, 0.2, 2.2, 0.8, 0.4, "EHR", EHR_C)
_box(ax, 1.2, 2.2, 0.8, 0.4, "ECG", ECG_C)
ax.bar([0.6, 1.6], [0.7, 0.3], width=0.5, bottom=1.1, color=[EHR_C, ECG_C], alpha=0.6)
ax.text(1.1, 1.95, "weights sum to 1", ha="center", fontsize=8)
_box(ax, 0.5, 0.55, 1.2, 0.35, "weighted sum", FUSED_C)
ax.text(1.1, 0.25, "weights depend on the input;\nyou can inspect them", ha="center", va="top", fontsize=8, color="#52514e")

ax = axes[4]
for px, py in demo.normal(0, 0.18, (4, 2)) + np.array([[0.6, 2.2], [1.5, 2.3], [0.8, 1.2], [1.6, 1.3]]):
    ax.scatter(px, py, color=ECG_C, s=40)
    ax.scatter(px + 0.12, py - 0.1, color=EHR_C, s=40, marker="s")
    ax.plot([px, px + 0.12], [py, py - 0.1], color="#52514e", lw=0.8)
ax.text(1.1, 0.25, "train encoders so matching\npairs land close (e.g. CLIP)", ha="center", va="top", fontsize=8, color="#52514e")
fig.suptitle("Orange: EHR embedding. Blue: ECG embedding. Green: fused.", x=0.01, ha="left", fontsize=10)
plt.tight_layout()
plt.show()

`FusionNet` encodes each modality to 16 numbers and combines them with one of
these operators. The ECG input is reduced to 32 principal components (PCA),
fitted inside each fold. Concatenation, addition and attention are written for you.

`cv_fusion_net` takes `n_pca`, `epochs` and `seeds` arguments, and `FusionNet`
takes the embedding size `d`. Once gating works, try changing them.

### TODO 5: gating

Complete the `"gate"` branch. Compute a gate `g` from both embeddings (use
`self.gate`, which maps the concatenated embeddings to 16 numbers, and a
sigmoid), then mix: `g * h_ehr + (1 - g) * h_ecg`.

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, ehr_dim, ecg_dim, mode, d=16):
        super().__init__()
        self.mode = mode
        self.enc_ehr = nn.Sequential(nn.Linear(ehr_dim, d), nn.ReLU())
        self.enc_ecg = nn.Sequential(nn.Dropout(0.2), nn.Linear(ecg_dim, d), nn.ReLU())
        self.gate = nn.Linear(2 * d, d)
        self.query = nn.Parameter(torch.randn(d) / d ** 0.5)
        self.key = nn.Linear(d, d)
        self.head = nn.Linear(2 * d if mode == "concat" else d, 1)

    def forward(self, ehr, ecg):
        h_ehr, h_ecg = self.enc_ehr(ehr), self.enc_ecg(ecg)
        if self.mode == "concat":
            z = torch.cat([h_ehr, h_ecg], dim=-1)
        elif self.mode == "add":
            z = h_ehr + h_ecg
        elif self.mode == "gate":
            # TODO: your code here
            raise NotImplementedError("Complete the TODO above")
        elif self.mode == "attention":
            tokens = torch.stack([h_ehr, h_ecg], dim=1)
            weights = torch.softmax(self.key(tokens) @ self.query / tokens.shape[-1] ** 0.5, dim=1)
            self.attention_weights = weights.detach()
            z = (weights.unsqueeze(-1) * tokens).sum(dim=1)
        return self.head(z).squeeze(-1)


def cv_fusion_net(mode, ecg_features, n_pca=32, epochs=100, seeds=3):
    """Out-of-fold probabilities from FusionNet, averaged over a few random seeds."""
    oof = np.zeros(len(y))
    attention = np.zeros((len(y), 2))
    as_tensor = lambda a: torch.tensor(a, dtype=torch.float32)
    for train, test in FOLDS:
        ehr_scaler = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()).fit(X_ehr[train])
        ehr_train, ehr_test = ehr_scaler.transform(X_ehr[train]), ehr_scaler.transform(X_ehr[test])
        ecg_scaler = StandardScaler().fit(ecg_features[train])
        ecg_train, ecg_test = ecg_scaler.transform(ecg_features[train]), ecg_scaler.transform(ecg_features[test])
        if n_pca:
            pca = PCA(n_pca, random_state=SEED).fit(ecg_train)
            ecg_train, ecg_test = pca.transform(ecg_train), pca.transform(ecg_test)
            pca_scaler = StandardScaler().fit(ecg_train)
            ecg_train, ecg_test = pca_scaler.transform(ecg_train), pca_scaler.transform(ecg_test)
        H_tr, H_te, E_tr, E_te = map(as_tensor, (ehr_train, ehr_test, ecg_train, ecg_test))
        y_tr = as_tensor(y[train])
        preds = []
        for seed in range(seeds):
            torch.manual_seed(seed)
            net = FusionNet(H_tr.shape[1], E_tr.shape[1], mode)
            opt = torch.optim.AdamW(net.parameters(), lr=3e-3, weight_decay=1e-2)
            for _ in range(epochs):
                net.train()
                loss = F.binary_cross_entropy_with_logits(net(H_tr, E_tr), y_tr)
                opt.zero_grad()
                loss.backward()
                opt.step()
            net.eval()
            with torch.no_grad():
                preds.append(torch.sigmoid(net(H_te, E_te)).numpy())
                if mode == "attention":
                    attention[test] += net.attention_weights.numpy() / seeds
        oof[test] = np.mean(preds, axis=0)
    return oof, attention


torch.manual_seed(SEED)
toy_net = FusionNet(N_EHR, 8, "gate").eval()
toy_net.head = nn.Identity()  # so the output is the fused vector itself
toy_ehr, toy_ecg = torch.randn(5, N_EHR), torch.randn(5, 8)
with torch.no_grad():
    z = toy_net(toy_ehr, toy_ecg)
    h_ehr, h_ecg = toy_net.enc_ehr(toy_ehr), toy_net.enc_ecg(toy_ecg)
check("each fused value lies between the EHR and ECG values",
      bool(((z >= torch.minimum(h_ehr, h_ecg) - 1e-6) & (z <= torch.maximum(h_ehr, h_ecg) + 1e-6)).all()),
      "a gate g between 0 and 1 mixes the two: g * h_ehr + (1 - g) * h_ecg")
check("the mix depends on the input, not a fixed half and half",
      not torch.allclose(z, (h_ehr + h_ecg) / 2),
      "compute g from both embeddings with self.gate and a sigmoid")

for mode in ["concat", "add", "gate", "attention"]:
    p, attention = cv_fusion_net(mode, X_ecg)
    record(f"Intermediate fusion ({mode})", p)
print(f"Average attention weight: EHR {attention[:, 0].mean():.2f}, ECG {attention[:, 1].mean():.2f}")

**Shared space (learnt).** In Part 1 we trained ECGFounder's projection so that
ECGs land close to their reports. Use that aligned 128-d embedding as the ECG
input instead. (The projection never saw the mortality labels.)

In [ ]:
with torch.no_grad():
    founder_clip.cpu().eval()
    X_ecg_shared = F.normalize(founder_clip.ecg_proj(torch.tensor(ecgfounder)), dim=-1).numpy()
p_shared, _ = cv_fusion_net("concat", X_ecg_shared)
record("Intermediate fusion (shared space)", p_shared)

**Questions for Part 3**

1. Rank the fusion methods. Which differences are bigger than the confidence
   intervals? How did your way of shrinking the ECG in TODO 4 compare with your
   neighbours'?
2. Gating could, in principle, trust the ECG more for patients without recent
   labs, which a fixed average cannot. Why does it not beat simple late fusion
   here? Think about how many patients we have.
3. What do the attention weights tell you? Is that the same as "the ECG doesn't
   matter"?
4. Suppose a third of patients have no ECG at test time. Which approaches cope
   most easily, and how?

*Your answers:*

## Part 4: a quick check on your features (5 min)

Fusion is only as good as its inputs. Two colleagues each suggest a change to
the EHR model:

- **A:** "Half the patients have no recent labs. Take each test's latest result
  from the whole admission instead, and nearly everyone has one."
- **B:** "Add `discharge_end_of_life_code`: 1 if the admission's discharge
  diagnoses include do not resuscitate (ICD-10 Z66) or palliative care (Z51.5)."

In [ ]:
whole_admission = last_lab_values(-np.inf, np.inf)
X_ehr_a = np.column_stack([X_ehr[:, :2], whole_admission, whole_admission.isna().all(axis=1)])
X_ehr_b = np.column_stack([X_ehr, meta["discharge_end_of_life_code"]])
for name, X in [("A: labs from the whole admission", X_ehr_a), ("B: EHR + end-of-life code", X_ehr_b)]:
    print(f"{name:34s} AUROC %.3f [%.3f, %.3f]" % auroc_ci(y, cv_predict(X, C=1.0)))

Without any ECG, each matches all our fusion work. **Question:** could you use
either model to predict mortality at the time of the ECG? When was each
feature recorded?

*Your answers:*

## Summary

In [ ]:
names = list(results)
auc = np.array([results[n][0] for n in names])
err = np.array([[results[n][0] - results[n][1], results[n][2] - results[n][0]] for n in names]).T
colours = ["#eb6834" if n == "EHR only" else "#2a78d6" if n.startswith("ECG") else "#1baf7a" for n in names]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(names, auc, xerr=err, color=colours)
ax.axvline(0.5, color="grey", lw=0.8, ls="--")
ax.set_xlim(0.45, 0.9)
ax.invert_yaxis()
ax.set_xlabel("AUROC, 1-year mortality (95% CI)")
plt.tight_layout()
plt.show()

**Take-home points**

- Pretraining at scale is what makes a modality useful: 800 pairs are not
  enough, but a foundation model's frozen features plus two linear layers are.
- Compare every fusion method with the best single modality, with confidence
  intervals.
- Where and how you fuse both matter: naive concatenation can hurt, and with
  little data, simple late fusion is hard to beat.
- And check when each feature was recorded before trusting a result.

**If you finish early**

There are two kinds of extension. Each one under *Going further* has a clear
question to answer in about 15 minutes. The *Open-ended* ones are bigger.
Whatever you try, keep the patient-level folds, fit everything inside the
training fold, and report AUROC with a confidence interval (use `record`, then
rerun the summary plot). Write down what you found and why in the answer box
below.

*Going further*

1. **Zero-shot AF detection.** Using `founder_clip`, average the projected
   report embeddings of the training ECGs whose reports mention atrial
   fibrillation, and separately of those that do not. Score each test ECG by
   its similarity to the AF average minus the non-AF one. How close does this
   get to the linear probe, which used AF labels? Why does it work at all?
2. **Learn the late-fusion weights (stacking).** Instead of a plain average,
   fit a logistic regression on the log-odds of `p_ehr` and `p_ecg`. The
   weights must be learnt from predictions on the training fold only, which
   themselves must be out-of-fold. Does it beat the plain average? Why, or why
   not?
3. **Missing ECGs.** Hide the ECG for a random third of patients at test time.
   Compare late fusion (fall back to `p_ehr` for them) with your early-fusion
   model from TODO 4 (fill their ECG features with the training mean). Which
   copes better, and why?
4. **Trends, not just the last value.** For each panel test, use the 72 hours
   before the ECG to compute the first, last, minimum and maximum result, the
   number of results and the change (last minus first). Do they beat the last
   value alone? Explain what you find.
5. **Does the ECG know the labs?** A high potassium (above 5.0 mmol/L) and
   poor kidney function (creatinine above 1.5 mg/dL) both change the ECG.
   Among ECGs with a recent result, fit cross-validated linear probes on
   `ecgfounder` to detect each. What does that say about how much the two
   modalities overlap?

*Open-ended*

6. **The lab history as a sequence.** `lab_sequences()` below turns each
   patient's results before the ECG into a padded sequence of blood draws.
   Write a small GRU encoder for it (`nn.GRU(19, 16, batch_first=True)`, with
   `nn.utils.rnn.pack_padded_sequence(..., enforce_sorted=False)`, and take
   the final hidden state). Use it in place of `enc_ehr` in `FusionNet`,
   keeping age and sex as a separate input. Does it beat the summary features
   from extension 4?
7. **Robust gating.** While training `FusionNet`, replace `h_ecg` with zeros
   for a random 30% of patients in each epoch (modality dropout). Then repeat
   extension 3 with the gated model. Does it cope as well as late fusion?
8. **Your own question.** Agree it with a demonstrator first.

*Your answers:*

In [ ]:
def lab_sequences(max_steps=8, hours=72):
    """Each ECG's panel results in the `hours` before it, as a padded sequence of blood draws.

    Returns a tensor (N, max_steps, 19) and the length of each sequence. Each step
    is one draw: the nine results (z-scored, 0 if not measured), nine flags for
    which tests were measured, and the time before the ECG in days. Only the
    latest max_steps draws are kept, and an ECG with no draws gets one empty step.
    """
    window = labs[labs["itemid"].isin(LAB_PANEL) & labs["hours_from_ecg"].between(-hours, 0)]
    draws = window.pivot_table(index=["pair_id", "hours_from_ecg"], columns="itemid", values="value", aggfunc="last")
    draws = draws.reindex(columns=list(LAB_PANEL))
    # z-score each test over all results; this uses no outcome information
    draws = (draws - draws.mean()) / draws.std()
    row_of = {pid: i for i, pid in enumerate(meta["pair_id"])}
    seqs = torch.zeros(len(meta), max_steps, 2 * len(LAB_PANEL) + 1)
    lengths = torch.ones(len(meta), dtype=torch.long)
    for pid, d in draws.groupby(level="pair_id"):
        d = d.tail(max_steps)
        days_before = -d.index.get_level_values("hours_from_ecg").to_numpy() / 24
        steps = np.column_stack([d.fillna(0), d.notna(), days_before])
        seqs[row_of[pid], :len(d)] = torch.tensor(steps, dtype=torch.float32)
        lengths[row_of[pid]] = len(d)
    return seqs, lengths


lab_seqs, lab_lengths = lab_sequences()
print(lab_seqs.shape, "| ECGs with at least 2 draws:", int((lab_lengths >= 2).sum()))